# Generator notebook

This notebook is used to prototype the generator python application.

The cell structure is intended to replicate the general structure of the generator script.

1. Import libraries and set basic variables
2. Load the configuration
3. Load the base demand
4. Execute transformations
5. Write output

In [2]:
# 1. Import libraries and set basic variables

import sys
import pandas as pd
import geopandas as gpd
from pathlib import Path
import json
from datetime import datetime

root_path = Path(globals()['_dh'][0]).resolve().parent
sys.path.append(str(root_path))

from paths import config_path, input_path, api_path
from library.utilities import get_path

In [39]:
# 2. Load the configuration

with (config_path / 'prototype.json').open('r', encoding='utf-8') as file:
    config = json.load(file)

In [4]:
# 3. Load the base demand

base_demand = pd.read_csv(input_path / config['loader']['properties']['access'] / 'base_demand' / get_path(config['loader']['name'],config['loader']['properties'], "csv"), index_col=['timestamp'], parse_dates=['timestamp'])

In [5]:
# 4 Execute transformations - Sort the transformations array

transformers = config['transformers']
transformers.sort(key=lambda x: x["order"])

In [6]:
# 4.1. Execute transformations - Split base demand (national) into municipalities

input = {input['name']: pd.read_csv(input_path / transformers[0]['access'] / input['path'], usecols=input['columns'], dtype={input['index']: str}, index_col=[input['index']]) for input in transformers[0]['inputs']}

municipality_demand = pd.DataFrame(
    base_demand['Sweden'].values[:, None] * input['municipality_energy_split']['ratio'].values, 
    index=base_demand.index, 
    columns=input['municipality_energy_split'].index
)

In [7]:
# 4.2. Execute transformations - Apply growth over time

# TODO: Do this using the config file instead

target_energy = 330 # TWh projected for 2045
yearly_growth = (target_energy/(base_demand['Sweden'].sum()/1_000_000))**(1/20) - 1 # About 4.32% growth yearly

ten_yrs_in_hours = (datetime(2045, 1, 1, 0, 0, 0) - datetime(2025, 1, 1, 0, 0, 0)).total_seconds() / 3600 # The number of hours between 2025-01-01 00:00:00 and 2046-01-01 00:00:00
hourly_growth = (target_energy/(base_demand['Sweden'].sum()/1_000_000))**(1/ten_yrs_in_hours) - 1

start_time = pd.Timestamp(f"2025-01-01 00:00:00")
end_time = pd.Timestamp(f"2045-12-31 23:00:00")
new_timestamps = pd.date_range(start=start_time, end=end_time, freq='h')
new_timestamps = new_timestamps[~((new_timestamps.month == 2) & (new_timestamps.day == 29))] # Skipping leap days

extended_demand = pd.DataFrame(index=municipality_demand.index.union(new_timestamps), columns=municipality_demand.columns)
extended_demand.index.name = municipality_demand.index.name

extended_demand.loc[base_demand.index] = municipality_demand # Insert the base municipality demand

for timestamp in new_timestamps:
    if timestamp >= start_time:
        hours_elapsed = (timestamp - start_time).total_seconds() / 3600
        growth_factor = (1 + hourly_growth) ** hours_elapsed
        base_values = municipality_demand.loc[timestamp.replace(year=start_time.year-1), :]  # Starting point
        extended_demand.loc[timestamp] = base_values * growth_factor


In [ ]:
# 5. Write output

# Write yearly per municipality (1h, 3h, 1d, 1w, 1m), for the country as a whole

# TODO: This script currently takes 14m14s to run through 20 years and 290 municipalities and 5 resolutions. I need to improve this.

resolutions = ['1h', '3h', '1d', '1W', '1ME']

def output(year):
    yearly_demand = extended_demand[extended_demand.index.year == year]
    resampled_yearly_demand = {res: yearly_demand.resample(res).sum() for res in resolutions}

    for resolution in resolutions:
        yearly_demand.sum(axis=1).resample(resolution).sum().to_csv(
            api_path / f"demand_t,geography={'00'},resolution={resolution},year={year}.csv.gz",
            header=['total demand (MWh)'],
            compression='gzip'
        )

    yearly_demand.sum().to_csv(api_path / f"demand,year={year}.csv.gz", header=['total demand (MWh)'], compression='gzip')

    for geo in extended_demand.columns:

        for resolution in resolutions:
            resampled_yearly_demand[resolution][geo].to_csv(
                api_path / f"demand_t,geography={geo},resolution={resolution},year={year}.csv.gz",
                header=['total demand (MWh)'],
                compression='gzip'
            )

for year in range(config['start-year'], config['end-year']):
    output(year)
    print(f"{year} is done!")


2025 is done!
2026 is done!
2027 is done!
2028 is done!
2029 is done!
2030 is done!
2031 is done!
2032 is done!
2033 is done!
2034 is done!
2035 is done!
2036 is done!
2037 is done!
2038 is done!
2039 is done!
2040 is done!
2041 is done!
2042 is done!
2043 is done!
2044 is done!


In [ ]:
# TODO: Include this above

municipalities_geojson = gpd.read_file(input_path / 'public' / 'geographies' / 'georef-sweden-kommun@public.geojson', encoding='utf-8')

municipalities = municipalities_geojson[['year', 'kom_code', 'kom_name', 'geometry']].copy()

for year in range(config['start-year'], config['end-year']):
    yearly_demand = extended_demand[extended_demand.index.year == year].sum()
    municipalities['year'] = year
    municipalities['demand'] = municipalities['kom_code'].map(yearly_demand).astype(float)
    municipalities['demand_unit'] = 'MWh'

    municipalities.to_file(api_path / f"demand_geo,year={year}.geojson", driver="GeoJSON", encoding="utf-8")
    

In [51]:
# TODO: Include this above

# Write the parameters.json

geographies_gdp = gpd.read_file(input_path / 'public' / 'geographies' / 'georef-sweden-kommun@public.geojson', encoding='utf-8')

geographies = geographies_gdp[['kom_type', 'kom_code', 'kom_name', 'lan_code', 'lan_name']]

geographies = geographies.rename(columns={
    "kom_type": "type",
    "kom_code": "id",
    "kom_name": "name",
    "lan_code": "parent_id",
    "lan_name": "parent_name"
})

new_row = pd.DataFrame({
    "type": ["Land"],
    "id": ["00"],
    "name": ["Sverige"],
    "parent_name": [""],
    "parent_id": [""]
})

geographies = pd.concat([geographies, new_row], ignore_index=True)

parameters = {
    'years': list(range(config['start-year'], config['end-year'])),
    'geographies': geographies.to_dict(orient="records"),
    'resolutions': config['output']['properties']['resolutions']
}

(api_path / "parameters.json").write_text(json.dumps(parameters, indent=4, ensure_ascii=False), encoding='utf-8')

53910

In [45]:
geographies

,type,id,name,parent_id,parent_name
0,Kommun,2506,Arjeplog,25,Norrbottens län
1,Kommun,0125,Ekerö,01,Stockholms län
2,Kommun,1762,Munkfors,17,Värmlands län
3,Kommun,1285,Eslöv,12,Skåne län
4,Kommun,2284,Örnsköldsvik,22,Västernorrlands län
...,...,...,...,...,...
285,Kommun,0184,Solna,01,Stockholms län
286,Kommun,1984,Arboga,19,Västmanlands län
287,Kommun,1280,Malmö,12,Skåne län
288,Kommun,0140,Nykvarn,01,Stockholms län


In [ ]:
# Generate some

max_values = []
min_values = []

for year in range(config['start-year'], config['end-year']):
    municipal_totals = extended_demand[extended_demand.index.year == year].sum()
    max_values.append(municipal_totals.max())
    min_values.append(municipal_totals.min())

35555.016596496724

In [26]:
print(min(min_values))
print(max(max_values))

35555.016596496724
15942725.686770272
